# Zomato Delhi — Tính toán KPI

Tính các KPI chính thức (Revenue, AOV, % khuyến mãi...) và xuất bảng tổng hợp. Cần chạy 02a_zomato_cleaning trước.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

## ĐƯỜNG DẪN

In [2]:
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().resolve().parent
CLEAN_PATH = BASE_DIR / 'data' / 'clean' / 'zomato_clean.csv'
KPI_DIR = BASE_DIR / 'outputs' / 'kpi_summary'
KPI_DIR.mkdir(parents=True, exist_ok=True)

## 1. ĐỌC DỮ LIỆU ĐÃ LÀM SẠCH

In [3]:
# Cần chạy 02a_zomato_cleaning trước để có file này
df = pd.read_csv(CLEAN_PATH, parse_dates=['Order Placed At'])
df['has_discount'] = df['has_discount'].astype(bool)
print("Shape dữ liệu đã làm sạch:", df.shape)

# Chỉ tính Revenue/AOV trên đơn thành công (Delivered) - theo xác nhận đã chốt
df_delivered = df[df['Order Status'] == 'Delivered'].copy()
print(f"Số đơn 'Delivered' dùng để tính KPI: {len(df_delivered)} / {len(df)}")

Shape dữ liệu đã làm sạch: (21321, 27)
Số đơn 'Delivered' dùng để tính KPI: 21131 / 21321


## 2. KPI CƠ BẢN: ORDERS, REVENUE, AOV

In [4]:
total_orders_delivered = len(df_delivered)
total_revenue = df_delivered['Total'].sum()
aov = df_delivered['Total'].mean()
pct_rating_available = df['Rating'].notna().mean() * 100

print(f"Total Orders (Delivered): {total_orders_delivered:,}")
print(f"Total Revenue: ₹{total_revenue:,.0f}")
print(f"AOV: ₹{aov:,.2f}")
print(f"Kiểm tra công thức Revenue = Orders × AOV: "
      f"{total_orders_delivered} × {aov:.2f} ≈ {total_orders_delivered * aov:,.0f} "
      f"(so với Total Revenue thực = {total_revenue:,.0f})")

Total Orders (Delivered): 21,131
Total Revenue: ₹14,423,380
AOV: ₹682.57
Kiểm tra công thức Revenue = Orders × AOV: 21131 × 682.57 ≈ 14,423,380 (so với Total Revenue thực = 14,423,380)


## 3. KPI KHUYẾN MÃI

In [5]:
pct_orders_with_discount = df_delivered['has_discount'].mean() * 100
aov_by_discount = df_delivered.groupby('has_discount')['Total'].mean()
aov_with_discount = aov_by_discount.get(True, np.nan)
aov_without_discount = aov_by_discount.get(False, np.nan)
corr_discount_bill = df_delivered['total_discount'].corr(df_delivered['Bill subtotal'])

print(f"\n% đơn có khuyến mãi: {pct_orders_with_discount:.1f}%")
print(f"AOV có khuyến mãi: ₹{aov_with_discount:,.2f} | AOV không khuyến mãi: ₹{aov_without_discount:,.2f}")


% đơn có khuyến mãi: 61.1%
AOV có khuyến mãi: ₹665.11 | AOV không khuyến mãi: ₹710.02


## 4. KPI VẬN HÀNH

In [6]:
avg_kpt = df['KPT duration (minutes)'].mean()
avg_rider_wait = df['Rider wait time (minutes)'].mean()
pct_delivered = (df['Order Status'] == 'Delivered').mean() * 100

## 5. TỔNG HỢP VÀ LƯU KPI SUMMARY

In [7]:
kpi_summary = {
    'dataset': 'Zomato',
    'market': 'India (Delhi NCR)',
    'total_orders_delivered': total_orders_delivered,
    'total_revenue': total_revenue,
    'aov': aov,
    'pct_orders_with_discount': pct_orders_with_discount,
    'aov_with_discount': aov_with_discount,
    'aov_without_discount': aov_without_discount,
    'corr_discount_vs_bill_subtotal': corr_discount_bill,
    'pct_rating_available': pct_rating_available,
    'avg_kpt_minutes': avg_kpt,
    'avg_rider_wait_minutes': avg_rider_wait,
    'pct_orders_delivered_successfully': pct_delivered,
}
pd.DataFrame([kpi_summary]).to_csv(KPI_DIR / 'zomato_kpi_summary.csv', index=False)

print("\n=== HOÀN TẤT TÍNH KPI ZOMATO ===")
print("Đã lưu outputs/kpi_summary/zomato_kpi_summary.csv")


=== HOÀN TẤT TÍNH KPI ZOMATO ===
Đã lưu outputs/kpi_summary/zomato_kpi_summary.csv
